# Raw API Calls

Direct `requests` calls to each API — **no collector classes, no database**.
Just the HTTP request and the raw JSON it returns, so you can see exactly what each source gives you.

Run the **imports** cell first, then any source cell.

In [9]:
# === Imports (run once) ===
import requests
import json

def show(data, n=2):
    """Pretty-print the first n records of a JSON response."""
    print(json.dumps(data[:n] if isinstance(data, list) else data, indent=2)[:20000])

## Senate LDA — lobbying filings (no key)

Endpoint: `https://lda.senate.gov/api/v1/filings/`

In [10]:
# === Senate LDA ===
resp = requests.get(
    "https://lda.senate.gov/api/v1/filings/",
    params={"filing_year": 2024, "page": 15, "page_size": 15},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
print("Total available:", data["count"])
show(data["results"])

Status: 200
Total available: 96854
[
  {
    "url": "https://lda.senate.gov/api/v1/filings/c41a6f36-9b5a-441f-a6c3-bf85c7c939be/",
    "filing_uuid": "c41a6f36-9b5a-441f-a6c3-bf85c7c939be",
    "filing_type": "RR",
    "filing_type_display": "Registration",
    "filing_year": 2024,
    "filing_period": "first_quarter",
    "filing_period_display": "1st Quarter (Jan 1 - Mar 31)",
    "filing_document_url": "https://lda.senate.gov/filings/public/filing/c41a6f36-9b5a-441f-a6c3-bf85c7c939be/print/",
    "filing_document_content_type": "text/html",
    "income": null,
    "expenses": null,
    "expenses_method": null,
    "expenses_method_display": null,
    "posted_by_name": "Richard Grafmeyer",
    "dt_posted": "2024-02-01T11:06:58-05:00",
    "termination_date": null,
    "registrant_country": "United States of America",
    "registrant_ppb_country": null,
    "registrant_address_1": "101 Constitution Avenue, NW Suite 675 East",
    "registrant_address_2": null,
    "registrant_different

## FEC — committees / Super PACs (DEMO_KEY)

Endpoint: `https://api.open.fec.gov/v1/committees`

In [3]:
# === FEC committees ===
resp = requests.get(
    "https://api.open.fec.gov/v1/committees",
    params={"api_key": "DEMO_KEY", "committee_type": "O", "per_page": 5, "page": 1},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
print("Total available:", data["pagination"]["count"])
show(data["results"])

Status: 200
Total available: 8524
[
  {
    "affiliated_committee_name": "NONE",
    "candidate_ids": [],
    "committee_id": "C00877886",
    "committee_type": "O",
    "committee_type_full": "Super PAC (Independent Expenditure-Only)",
    "cycles": [
      2024,
      2026
    ],
    "designated_agent_city": "TALLAHASSEE",
    "designated_agent_first_name": "SHELBY",
    "designated_agent_last_name": "GREEN",
    "designated_agent_middle_name": null,
    "designated_agent_name": "GREEN, SHELBY",
    "designated_agent_phone_number": "8506613941",
    "designated_agent_prefix": null,
    "designated_agent_state": "FL",
    "designated_agent_street1": "2800 S ADAMS ST UNIT 5651",
    "designated_agent_street2": null,
    "designated_agent_suffix": null,
    "designated_agent_title": "TREASURER",
    "designated_agent_zip": "32314",
    "designation": "U",
    "designation_full": "Unauthorized",
    "filing_frequency": "Q",
    "first_f1_date": "2024-05-02",
    "first_file_date": "2024-

## FEC — disbursements / spending (DEMO_KEY)

Endpoint: `https://api.open.fec.gov/v1/schedules/schedule_b`

In [11]:
# === FEC disbursements ===
resp = requests.get(
    "https://api.open.fec.gov/v1/schedules/schedule_b",
    params={"api_key": "DEMO_KEY", "two_year_transaction_period": 2024, "per_page": 5, "page": 1},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
show(data["results"])

Status: 200
[
  {
    "amendment_indicator": "A",
    "amendment_indicator_desc": "ADD",
    "back_reference_schedule_id": "SB17",
    "back_reference_transaction_id": "B664BDFCC4E9C439A854",
    "beneficiary_committee_name": null,
    "candidate_first_name": null,
    "candidate_id": null,
    "candidate_last_name": null,
    "candidate_middle_name": null,
    "candidate_name": null,
    "candidate_office": null,
    "candidate_office_description": null,
    "candidate_office_district": null,
    "candidate_office_state": null,
    "candidate_office_state_full": null,
    "candidate_prefix": null,
    "candidate_suffix": null,
    "category_code": null,
    "category_code_full": null,
    "comm_dt": null,
    "committee": {
      "affiliated_committee_name": "TEAM CELESTE",
      "candidate_ids": [
        "H4UT02296"
      ],
      "city": "CEDAR CITY",
      "committee_id": "C00842765",
      "committee_type": "H",
      "committee_type_full": "House",
      "cycle": 2024,
      "cy

## Congress.gov — bills (needs a key)

Get a free key at https://api.congress.gov/sign-up/ and paste it below.
Endpoint: `https://api.congress.gov/v3/bill/{congress}/{type}`

In [12]:
# === Congress.gov bills ===
CONGRESS_API_KEY = "PASTE_YOUR_KEY_HERE"

resp = requests.get(
    "https://api.congress.gov/v3/bill/118/hr",
    params={"api_key": CONGRESS_API_KEY, "format": "json", "limit": 5},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
show(data["bills"])

Status: 403


KeyError: 'bills'

# Amendment text — and the Congressional Record

**The key thing to understand before doing NLP on amendments:**

The amendment text endpoint is `/v3/amendment/{congress}/{type}/{number}/text`, but coverage is uneven:

| Case | What you get |
|------|--------------|
| **Senate amendments, 117th Congress (2021) → now** | Full text directly in the API (`textVersions` → HTML/PDF) ✅ clean |
| House amendments (any congress) | **Only a link into the Congressional Record** — no text field |
| Senate amendments **before** the 117th | **Only a link into the Congressional Record** |
| Some House amendments 117th+ | No text granule available at all |

So there are **two paths** to amendment text:
1. **API-native** (easy, clean) — only Senate amendments 117th+.
2. **Congressional Record** (harder) — page-ranged granules (e.g. `S1234`–`S1240`) whose actual text lives as HTML/PDF on GovInfo.

**Scoping implication:** for the capstone, the clean spine is *Senate amendments, 117th Congress forward*. Older / House amendment text means scraping the Record and is a stretch goal.

> Set your key in the next cell to run these.

In [15]:
# === Amendment key + helper ===
# Paste your key here (same one as the bills cell).
CONGRESS_API_KEY = "HnmLbuDGTEu5P2PFJlZyA2fz0aba11f1ttzphc7B"
BASE = "https://api.congress.gov/v3"

def cg_get(path, **params):
    """GET a Congress.gov path and return JSON."""
    params = {"api_key": CONGRESS_API_KEY, "format": "json", **params}
    r = requests.get(f"{BASE}/{path.lstrip('/')}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

### Step 1 — list Senate amendments (117th), pick one

In [16]:
# === Step 1: list amendments ===
# SAMDT = Senate amendment. 117th Congress is the first with full text.
data = cg_get("amendment/117/samdt", limit=5)
for a in data["amendments"]:
    print(a["type"], a["number"], "-", a.get("purpose") or a.get("description") or "(no purpose)")

SAMDT 2 - (no purpose)
SAMDT 3 - (no purpose)
SAMDT 4 - (no purpose)
SAMDT 5 - (no purpose)
SAMDT 1 - In the nature of a substitute.


### Step 2 — get the text endpoint for one amendment

This returns `textVersions` → `formats` (HTML / PDF). For Senate 117th+ these are real text files.

In [ ]:
# === Step 2: amendment text endpoint ===
# Use a known Senate amendment with full text. SA 2137 (117th) on the Infrastructure bill.
CONGRESS, AMDT_TYPE, AMDT_NUM = 117, "samdt", 2137

text_data = cg_get(f"amendment/{CONGRESS}/{AMDT_TYPE}/{AMDT_NUM}/text")
versions = text_data.get("textVersions", [])
print(f"Text versions available: {len(versions)}\n")
for v in versions:
    print("type:", v.get("type"), "| date:", v.get("date"))
    for fmt in v.get("formats", []):
        print("   ", fmt.get("type"), "->", fmt.get("url"))

### Step 3 — download the actual text

Grab the HTML (or PDF) URL from above and fetch it. This is the raw amendment text you'd feed into NLP.

In [ ]:
# === Step 3: download the text ===
# Pick the first HTML format URL we found above.
html_url = None
for v in versions:
    for fmt in v.get("formats", []):
        if fmt.get("type") in ("HTML", "Formatted Text"):
            html_url = fmt["url"]
            break
    if html_url:
        break

if html_url:
    raw = requests.get(html_url, timeout=30).text
    print("Fetched", len(raw), "chars from:", html_url, "\n")
    print(raw[:1500])
else:
    print("No HTML format on this amendment — it likely only links to the Congressional Record.")

### Step 4 — the Congressional Record fallback (House / pre-117th)

When an amendment has **no `textVersions`**, its text only exists in the Congressional Record.
You navigate the Record by **volume → issue → articles**, and each article has page ranges
(`startPage`/`endPage`, e.g. `S1234`) plus text URLs (HTML/PDF granules hosted on GovInfo).

The amendment's `latestAction.text` usually names the page (e.g. "Submitted ... see Congressional Record S1234"),
which you match against article page ranges to find the right granule.

In [ ]:
# === Step 4: navigate the Congressional Record ===
# List recent issues to see the volume/issue structure, then drill into one issue's articles.
issues = cg_get("daily-congressional-record", limit=3)
for iss in issues["dailyCongressionalRecord"]:
    print("Vol", iss["volumeNumber"], "Issue", iss["issueNumber"], "-", iss["issueDate"])

# Drill into one issue's articles (each article = a chunk of the Record with page range + text URL).
vol = issues["dailyCongressionalRecord"][0]["volumeNumber"]
issue = issues["dailyCongressionalRecord"][0]["issueNumber"]
arts = cg_get(f"daily-congressional-record/{vol}/{issue}/articles")

print("\nArticles in Vol", vol, "Issue", issue, ":")
for section in arts.get("articles", []):
    print(" Section:", section.get("name"))
    for item in section.get("sectionArticles", [])[:3]:
        pages = f"{item.get('startPage')}-{item.get('endPage')}"
        url = (item.get("text") or [{}])[0].get("url")
        print("   ", pages, "|", item.get("title", "")[:60], "|", url)